# Relativistic Breit-Wigner convolved with detector resolution

This notebook demonstrates the generic one-dimensional convolution layer using the **same `RelativisticBreitWigner` implementation used by the Dalitz amplitude model**. We convert the complex lineshape into a normalized isolated-resonance intensity and smear that intensity with a Gaussian detector response.


In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    ConvolvedPDF1D, GaussianResolution1D, LineshapeIntensity1D,
    Parameter, RelativisticBreitWigner, ResonanceContext, enable_x64,
)

enable_x64()


## $K^*(892)^0\to K^+\pi^-$ with the DalitzPlotFitter RBW

The amplitude lineshape already implemented in the project is

$$R(m)=\frac{1}{m_0^2-m^2-i m_0\Gamma(m)},$$

with the running width

$$\Gamma(m)=\Gamma_0\left(\frac{q}{q_0}\right)^{2L+1}\frac{m_0}{m}X_L^2(q,q_0).$$

For this one-dimensional resolution example we use the isolated resonance intensity

$$f(m)=\frac{|R(m)|^2}{\int |R(m')|^2dm'}.$$

This reuses the same running width and Blatt-Weisskopf conventions as the amplitude model; no second Breit-Wigner formula is introduced.

The physical mass interval of the $K\pi$ pair is inferred directly from the resonance context:

$$m_{\min}=m_K+m_\pi,\qquad m_{\max}=m_B-m_{\pi_{\rm bachelor}}.$$


In [ ]:
context = ResonanceContext(
    parent_mass=5.27934,
    daughter_masses=(0.493677, 0.13957039),   # K+, pi-
    bachelor_mass=0.13957039,                 # bachelor pi+
    spin=1,
    pole_mass=0.8958,
    pole_width=0.0474,
    resonance_radius=4.0,
    parent_radius=4.0,
)

true_mass = LineshapeIntensity1D.from_context(
    RelativisticBreitWigner(),
    context,
    order=512,
)

print(f"Physical Kpi range: {true_mass.low:.6f} to {true_mass.high:.6f} GeV")

resolution = GaussianResolution1D(sigma=0.008)
reco_mass = ConvolvedPDF1D(
    true_mass, resolution,
    true_low=true_mass.low, true_high=true_mass.high,
    observed_low=true_mass.low, observed_high=true_mass.high,
    order=96,
)

mass = jnp.linspace(true_mass.low, true_mass.high, 3000)
plt.figure(figsize=(8,5))
plt.plot(mass, true_mass(mass), label="true relativistic RBW intensity")
plt.plot(mass, reco_mass(mass), label=r"RBW intensity $\otimes$ Gaussian")
plt.xlabel(r"$m(K\pi)$ [GeV]")
plt.ylabel("normalized density")
plt.legend()
plt.show()


In [ ]:
print("True intensity normalization:", float(jnp.trapezoid(true_mass(mass), mass)))
print("Observed normalization:", float(jnp.trapezoid(reco_mass(mass), mass)))
print("Fraction retained before observed-window renormalization:", float(reco_mass.normalization()))


## Vary the detector resolution

The pole width $\Gamma_0$ belongs to the resonance dynamics. The detector width $\sigma_{\rm res}$ is a separate quantity and may itself be a fit parameter.


In [ ]:
sigma_res = Parameter("kpi_resolution.sigma", 0.008, bounds=(0.001, 0.030))
bias_res = Parameter("kpi_resolution.bias", 0.0, bounds=(-0.010, 0.010))

floating_reco = ConvolvedPDF1D(
    true_mass,
    GaussianResolution1D(sigma=sigma_res, bias=bias_res),
    true_low=true_mass.low, true_high=true_mass.high,
    observed_low=true_mass.low, observed_high=true_mass.high,
    order=96,
)

narrow = floating_reco(mass, {"kpi_resolution.sigma":0.003, "kpi_resolution.bias":0.0})
nominal = floating_reco(mass, {"kpi_resolution.sigma":0.008, "kpi_resolution.bias":0.0})
broad = floating_reco(mass, {"kpi_resolution.sigma":0.020, "kpi_resolution.bias":0.003})

plt.figure(figsize=(8,5))
plt.plot(mass, true_mass(mass), label="true RBW intensity")
plt.plot(mass, narrow, label="sigma_res = 3 MeV")
plt.plot(mass, nominal, label="sigma_res = 8 MeV")
plt.plot(mass, broad, label="sigma_res = 20 MeV, bias = 3 MeV")
plt.xlim(0.75, 1.05)  # zoom only for visualization; PDF is normalized on full physical range
plt.xlabel(r"$m(K\pi)$ [GeV]")
plt.ylabel("normalized density")
plt.legend()
plt.show()


## What exactly is being convolved?

For an isolated resonance, the detector response acts on the measurable intensity, so the example convolves $|R(m)|^2$, not the complex amplitude $R(m)$ itself.

For a **coherent sum of interfering amplitudes**, the correct object is the full intensity

$$|\mathcal A(m)|^2=\left|\sum_r c_r A_r(m)\right|^2,$$

and the resolution kernel must act on that full intensity. Convolving each resonance intensity independently would lose interference terms.

Likewise, genuine Dalitz resolution is generally multidimensional and remains a migration-kernel problem rather than independent 1D convolutions in the invariant masses.


## The constant-width `BreitWigner1D` example is still useful

`BreitWigner1D` remains available as a simple normalized Lorentz/Cauchy PDF and gives the textbook Voigt-profile example. For amplitude-analysis physics, however, `LineshapeIntensity1D(RelativisticBreitWigner(), context, ...)` is the preferred demonstration because it reuses the project's actual resonance dynamics.
